<a href="https://colab.research.google.com/github/NIVETHA-NS/GENARATIVE-AI-AND-LARGE-LANGUAGE-MODELS_LAB_2026/blob/main/GEN_AI_EXP_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import faiss
import torch

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ==========================================
# 1. Knowledge Base
# ==========================================
documents = [
    "The Eiffel Tower is located in Paris, France and was completed in 1889.",
    "Retrieval-Augmented Generation (RAG) combines document retrieval with text generation.",
    "Python is a popular high-level programming language used in AI development.",
    "Vector databases store embeddings and support fast similarity search."
]

# ==========================================
# 2. Load Embedding Model
# ==========================================
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

doc_embeddings = embed_model.encode(
    documents,
    convert_to_numpy=True
).astype(np.float32)

# ==========================================
# 3. Create FAISS Index
# ==========================================
dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

# ==========================================
# 4. User Query
# ==========================================
query = "What is RAG in AI?"

query_embedding = embed_model.encode(
    [query],
    convert_to_numpy=True
).astype(np.float32)

k = 2

distances, indices = index.search(query_embedding, k)

retrieved_chunks = [documents[i] for i in indices[0]]

# ==========================================
# 5. Build Prompt
# ==========================================
context = "\n".join(retrieved_chunks)

prompt = f"""
Answer the question using the context below.

Context:
{context}

Question:
{query}

Answer:
"""

# ==========================================
# 6. Load FLAN-T5
# ==========================================
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Move model to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

inputs = {k: v.to(device) for k, v in inputs.items()}

# ==========================================
# 7. Generate Answer
# ==========================================
outputs = model.generate(
    **inputs,
    max_new_tokens=60,
    temperature=0.7,
    do_sample=False
)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

# ==========================================
# 8. Print Results
# ==========================================
print("=" * 50)
print("Retrieved Context:")
print("=" * 50)

for i, chunk in enumerate(retrieved_chunks, 1):
    print(f"{i}. {chunk}")

print("\n" + "=" * 50)
print("Generated Answer:")
print("=" * 50)
print(answer)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Retrieved Context:
1. Retrieval-Augmented Generation (RAG) combines document retrieval with text generation.
2. Python is a popular high-level programming language used in AI development.

Generated Answer:
combines document retrieval with text generation
